In [1]:
!pip install pandas numpy faker tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 20.0 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np

from faker import Faker
from tqdm import tqdm

import random
from datetime import datetime, timedelta

In [3]:
fake = Faker()

NUM_USERS = 10000
NUM_BOTS = 200

NUM_VIDEOS = 50000
NUM_COMMENTS = 500000
NUM_CLICKS = 2000000

START_DATE = datetime(2025, 1, 1)
END_DATE = datetime(2025, 3, 31)

In [4]:
users = []

for user_id in tqdm(range(1, NUM_USERS + 1)):

    is_bot = user_id <= NUM_BOTS

    created_at = fake.date_time_between(
        start_date=START_DATE,
        end_date=END_DATE
    )

    users.append({
        "user_id": user_id,
        "account_created": created_at,
        "country": fake.country(),
        "device_id": fake.uuid4(),
        "is_bot": is_bot
    })

users_df = pd.DataFrame(users)

users_df.head()

100%|██████████| 10000/10000 [00:00<00:00, 30888.67it/s]


,user_id,account_created,country,device_id,is_bot
0,1,2025-02-25 03:23:52.183625,Puerto Rico,e815f9ae-6cf0-40b3-9d32-5d7a8231a97d,True
1,2,2025-02-08 23:30:09.509617,Israel,4d15697f-5fc1-4020-adac-d5a35d468ecf,True
2,3,2025-01-17 15:14:10.379904,Chad,daa6e4a9-c54e-46c9-ab53-f3a0b232aaa5,True
3,4,2025-03-18 18:26:01.007076,Turks and Caicos Islands,75ef4f46-5b83-4289-90c8-fd7ff452763e,True
4,5,2025-03-08 21:34:57.496645,Brazil,cd71e88a-b598-4881-b83a-895f21301723,True


In [5]:
videos = []

for video_id in tqdm(range(1, NUM_VIDEOS + 1)):

    uploader = random.randint(1, NUM_USERS)

    upload_time = fake.date_time_between(
        start_date=START_DATE,
        end_date=END_DATE
    )

    videos.append({
        "video_id": video_id,
        "uploader_id": uploader,
        "upload_time": upload_time,
        "category": random.choice([
            "gaming",
            "music",
            "education",
            "sports",
            "tech"
        ])
    })

videos_df = pd.DataFrame(videos)

100%|██████████| 50000/50000 [00:00<00:00, 84803.83it/s]


In [6]:
spam_comments = [
    "Amazing content!",
    "Check my profile!",
    "Free followers here!",
    "Watch my channel!",
    "Best video ever!"
]

In [7]:
comments = []

for comment_id in tqdm(range(1, NUM_COMMENTS + 1)):

    user = random.randint(1, NUM_USERS)

    is_bot = user <= NUM_BOTS

    if is_bot:
        comment = random.choice(spam_comments)
    else:
        comment = fake.sentence()

    comments.append({
        "comment_id": comment_id,
        "user_id": user,
        "video_id": random.randint(1, NUM_VIDEOS),
        "comment_text": comment,
        "created_at": fake.date_time_between(
            start_date=START_DATE,
            end_date=END_DATE
        )
    })

comments_df = pd.DataFrame(comments)

100%|██████████| 500000/500000 [00:17<00:00, 29086.09it/s]


In [8]:
actions = [
    "view",
    "like",
    "share",
    "subscribe"
]

clicks = []

for click_id in tqdm(range(1, NUM_CLICKS + 1)):

    user = random.randint(1, NUM_USERS)

    clicks.append({
        "click_id": click_id,
        "user_id": user,
        "video_id": random.randint(1, NUM_VIDEOS),
        "action_type": random.choice(actions),
        "created_at": fake.date_time_between(
            start_date=START_DATE,
            end_date=END_DATE
        )
    })

clicks_df = pd.DataFrame(clicks)

100%|██████████| 2000000/2000000 [00:27<00:00, 72884.62it/s]


In [9]:
bot_ring_1 = list(range(1, 41))
bot_ring_2 = list(range(41, 81))
bot_ring_3 = list(range(81, 121))
bot_ring_4 = list(range(121, 161))
bot_ring_5 = list(range(161, 201))

In [10]:
target_videos = random.sample(
    range(1, NUM_VIDEOS + 1),
    100
)

In [11]:
ring_events = []

event_id = NUM_CLICKS + 1

for bot in range(1, 201):

    for video in target_videos:

        ring_events.append({
            "click_id": event_id,
            "user_id": bot,
            "video_id": video,
            "action_type": "like",
            "created_at": START_DATE + timedelta(
                minutes=random.randint(0, 5000)
            )
        })

        event_id += 1

ring_df = pd.DataFrame(ring_events)

clicks_df = pd.concat(
    [clicks_df, ring_df],
    ignore_index=True
)

In [12]:
vulnerability_events = []

for bot in range(1, 201):

    created = users_df.loc[
        users_df.user_id == bot,
        "account_created"
    ].iloc[0]

    for _ in range(300):

        vulnerability_events.append({
            "click_id": event_id,
            "user_id": bot,
            "video_id": random.randint(1, NUM_VIDEOS),
            "action_type": "share",
            "created_at": created + timedelta(
                minutes=random.randint(1, 60)
            )
        })

        event_id += 1

vuln_df = pd.DataFrame(vulnerability_events)

clicks_df = pd.concat(
    [clicks_df, vuln_df],
    ignore_index=True
)

In [13]:
users_df.to_csv("users.csv", index=False)
videos_df.to_csv("videos.csv", index=False)
comments_df.to_csv("comments.csv", index=False)
clicks_df.to_csv("clicks.csv", index=False)

In [14]:
from google.colab import files

files.download("users.csv")
files.download("videos.csv")
files.download("comments.csv")
files.download("clicks.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
print("Users:", users_df.shape)
print("Videos:", videos_df.shape)
print("Comments:", comments_df.shape)
print("Clicks:", clicks_df.shape)

Users: (10000, 5)
Videos: (50000, 4)
Comments: (500000, 5)
Clicks: (2080000, 5)
